In [1]:
import pandas as pd

dataset_path = r'C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc'
df = pd.read_parquet(dataset_path + '/domain_users.parquet')
print(df.head())

           group       username
0          seoul     timotlopez
1          seoul     timotlopez
2          seoul     timotlopez
3  domain admins     timotlopez
4  domain admins  Administrator


In [2]:
import os

for root, dirs, files in os.walk(dataset_path):
    for f in files:
        print(os.path.join(root, f))

C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\.DS_Store
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\domain_users.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\loggedonusers\0000DQQEE.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\loggedonusers\0001LXQEN.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\useraccounts\0000DQQEE.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\useraccounts\0001LXQEN.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\w32drivers\0000DQQEE.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\w32drivers\0001LXQEN.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-8448-42c6-a394-bd54c981eedc\w32persistence-fileitems\0000DQQEE.parquet
C:\Users\nahla\Desktop\threat_hunt_task\3d276079-

In [3]:
import glob

def load_all(folder_name):
    files = glob.glob(dataset_path + f'/{folder_name}/*.parquet')
    dfs = []
    for f in files:
        d = pd.read_parquet(f)
        d['hostname'] = os.path.basename(f).replace('.parquet', '')
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

In [4]:
processes = load_all('w32processes')
processes_memory = load_all('w32processes-memorysections')
services = load_all('w32services')
tasks = load_all('w32tasks')
drivers = load_all('w32drivers')
persistence_reg = load_all('w32persistence-registryitems')
persistence_files = load_all('w32persistence-fileitems')
persistence_svc = load_all('w32persistence-serviceitems')
logged_users = load_all('loggedonusers')
user_accounts = load_all('useraccounts')

print(processes.shape, processes_memory.shape, services.shape, tasks.shape)

(95, 6) (3274, 9) (997, 14) (253, 14)


In [5]:
print(logged_users.columns.tolist())
logged_users

['domain-user', 'hostname', 'local-user', 'username']


,domain-user,hostname,local-user,username
0,False,0000DQQEE,True,svchost.exe
1,False,0000DQQEE,True,winlogon.exe
2,False,0000DQQEE,True,RuntimeBroker.exe
3,False,0000DQQEE,True,lsass.exe
4,False,0000DQQEE,True,dwm.exe
5,False,0000DQQEE,True,conhost.exe
6,False,0000DQQEE,True,LogonUI.exe
7,False,0000DQQEE,True,spoolsv.exe
8,False,0000DQQEE,True,dllhost.exe
9,False,0000DQQEE,True,msdtc.exe


In [6]:
logged_users[logged_users.astype(str).apply(lambda row: row.str.contains('timotlopez|Administrator', case=False, na=False)).any(axis=1)]

,domain-user,hostname,local-user,username
17,True,0000DQQEE,False,timotlopez
47,True,0001LXQEN,False,timotlopez
55,True,0001LXQEN,False,Administrator


In [7]:
print(processes.columns.tolist())
processes[['hostname'] + [c for c in processes.columns if 'name' in c.lower() or 'path' in c.lower()]]

['arguments', 'hostname', 'name', 'path', 'pid', 'username']


,hostname,hostname,name,path,username
0,0000DQQEE,0000DQQEE,svchost.exe,C:\Windows\System32,NT AUTHORITY\SYSTEM
1,0000DQQEE,0000DQQEE,AvastSvc.exe,,timotlopez
2,0000DQQEE,0000DQQEE,dwm.exe,C:\Windows\system32,NT AUTHORITY\SYSTEM
3,0000DQQEE,0000DQQEE,dllhost.exe,C:\Windows\system32,NT AUTHORITY\SYSTEM
4,0000DQQEE,0000DQQEE,putty.exe,C:\Program Files\PuTTY,svc_lw
...,...,...,...,...,...
90,0001LXQEN,0001LXQEN,AvastSvc.exe,,timotlopez
91,0001LXQEN,0001LXQEN,CyberGhost.Service.exe,C:\Program Files\CyberGhost 6,timotlopez
92,0001LXQEN,0001LXQEN,dwm.exe,C:\Windows\system32,NT AUTHORITY\SYSTEM
93,0001LXQEN,0001LXQEN,svchost.exe,C:\Windows\system32,NT AUTHORITY\SYSTEM


In [8]:
suspicious = processes[processes.astype(str).apply(lambda row: row.str.contains('Temp|AppData|ProgramData', case=False, na=False)).any(axis=1)]
suspicious

,arguments,hostname,name,path,pid,username
45,"""C:\Users\timotlopez\AppData\Local\Microsoft\O...",0000DQQEE,OneDrive.exe,C:\Users\timotlopez\AppData\Local\Microsoft\On...,3037,timotlopez
48,"""C:\Users\timotlopez\AppData\Local\Microsoft\O...",0001LXQEN,OneDrive.exe,C:\Users\timotlopez\AppData\Local\Microsoft\On...,5043,timotlopez
79,C:\Windows\System32\cmd.exe /C C:\Users\timotl...,0001LXQEN,cmd.exe,C:\Windows\System32\cmd.exe,5612,NT AUTHORITY\SYSTEM


In [9]:
print(persistence_reg.columns.tolist())
persistence_reg

['hostname', 'keypath', 'path', 'text', 'type', 'username', 'valuename']


,hostname,keypath,path,text,type,username,valuename
0,0000DQQEE,Classes\CLSID\{2D3F8A1B-6DCD-4ED5-BDBA-A096594...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{2D3...,C:\Windows\System32\twinapi.dll,REG_SZ,NT AUTHORITY\SYSTEM,
1,0000DQQEE,Classes\CLSID\{d509c21a-b88c-4ad1-8fad-d6a7572...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{d50...,%SystemRoot%\system32\explorerframe.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
2,0000DQQEE,Classes\CLSID\{A6EE35C6-87EC-47DF-9F22-1D5AAD8...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{A6E...,%SystemRoot%\system32\windowscodecs.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
3,0000DQQEE,Classes\CLSID\{649EEC1E-B579-4E8C-BB3B-4997F84...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{649...,C:\Windows\System32\Dxtrans.dll,REG_SZ,NT AUTHORITY\SYSTEM,
4,0000DQQEE,Classes\CLSID\{8369AB20-56C9-11D0-94E8-00AA005...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{836...,C:\Windows\System32\occache.dll,REG_SZ,NT AUTHORITY\SYSTEM,
...,...,...,...,...,...,...,...
12168,0001LXQEN,Classes\CLSID\{B92E345D-F52D-41F3-B562-081BC77...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{B92...,%SystemRoot%\system32\windowscodecs.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
12169,0001LXQEN,Classes\CLSID\{3050f6cd-98b5-11cf-bb82-00aa00b...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{305...,C:\Windows\System32\iepeers.dll,REG_SZ,NT AUTHORITY\SYSTEM,
12170,0001LXQEN,Classes\CLSID\{b27b520e-46db-4720-b9c5-5f80aca...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{b27...,%SystemRoot%\system32\hgcpl.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
12171,0001LXQEN,Classes\CLSID\{9456A480-E88B-43EA-9E73-0B2D9B7...,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{945...,%SystemRoot%\system32\windowscodecs.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,


In [10]:
print(persistence_files.columns.tolist())
persistence_files

['drive', 'fileextension', 'filename', 'filepath', 'fullpath', 'hostname', 'username']


,drive,fileextension,filename,filepath,fullpath,hostname,username
0,c,dll,pwlauncher.dll,Windows\System32,c:\Windows\System32\pwlauncher.dll,0000DQQEE,NT AUTHORITY\SYSTEM
1,c,exe,OneDrive.exe,Users\$user\AppData\Local\microsoft\OneDrive,c:\Users\$user\AppData\Local\microsoft\OneDriv...,0000DQQEE,$domain\$user
2,C,dll,oleaut32.dll,Windows\System32,C:\Windows\System32\oleaut32.dll,0000DQQEE,NT AUTHORITY\SYSTEM
3,c,sys,rdpbus.sys,Windows\System32\drivers,c:\Windows\System32\drivers\rdpbus.sys,0000DQQEE,NT SERVICE\TrustedInstaller
4,c,dll,qoswmi.dll,Windows\System32\wbem,c:\Windows\System32\wbem\qoswmi.dll,0000DQQEE,NT AUTHORITY\SYSTEM
...,...,...,...,...,...,...,...
4591,c,dll,tspubwmi.dll,Windows\System32,c:\Windows\System32\tspubwmi.dll,0001LXQEN,NT AUTHORITY\SYSTEM
4592,c,dll,smartcardcredentialprovider.dll,Windows\System32,c:\Windows\System32\smartcardcredentialprovide...,0001LXQEN,NT AUTHORITY\SYSTEM
4593,c,sys,SerCx2.sys,Windows\System32\drivers,c:\Windows\System32\drivers\SerCx2.sys,0001LXQEN,NT AUTHORITY\SYSTEM
4594,c,SYS,VSTXRAID.SYS,Windows\System32\drivers,c:\Windows\System32\drivers\VSTXRAID.SYS,0001LXQEN,NT AUTHORITY\SYSTEM


In [11]:
print(persistence_svc.columns.tolist())
persistence_svc

[]


""


In [12]:
pd.set_option('display.max_colwidth', None)
processes[processes['pid'] == 5612]

,arguments,hostname,name,path,pid,username
79,C:\Windows\System32\cmd.exe /C C:\Users\timotlopez\AppData\Local\Microsoft\WindowsApps\run_pd.bat,0001LXQEN,cmd.exe,C:\Windows\System32\cmd.exe,5612,NT AUTHORITY\SYSTEM


In [13]:
persistence_reg[persistence_reg['hostname'] == '0001LXQEN']

,hostname,keypath,path,text,type,username,valuename
6087,0001LXQEN,Classes\CLSID\{4be5c3b3-53a2-4f2f-af9d-d26a5327eb92}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{4be5c3b3-53a2-4f2f-af9d-d26a5327eb92}\InProcServer32\,C:\Windows\System32\IME\IMEKR\APPLETS\imkrskf.dll,REG_SZ,NT AUTHORITY\SYSTEM,
6088,0001LXQEN,Classes\CLSID\{ed9d80b9-d157-457b-9192-0e7280313bf0}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{ed9d80b9-d157-457b-9192-0e7280313bf0}\InProcServer32\,%SystemRoot%\system32\zipfldr.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
6089,0001LXQEN,Wow6432Node\Classes\CLSID\{3F037241-414E-11D1-A7CE-00A0C913F73C}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Wow6432Node\Classes\CLSID\{3F037241-414E-11D1-A7CE-00A0C913F73C}\InProcServer32\,%SystemRoot%\System32\dmstyle.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
6090,0001LXQEN,Classes\CLSID\{46080CA7-7CB8-3A55-A72E-8E50ECA4D4FC}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{46080CA7-7CB8-3A55-A72E-8E50ECA4D4FC}\InProcServer32\,C:\Windows\System32\mscoree.dll,REG_SZ,NT AUTHORITY\SYSTEM,
6091,0001LXQEN,Classes\CLSID\{0C1A0EF4-B44F-4179-BDDD-31C27DE7A6D1}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{0C1A0EF4-B44F-4179-BDDD-31C27DE7A6D1}\InProcServer32\,%SystemRoot%\System32\InternetMailCsp.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
...,...,...,...,...,...,...,...
12168,0001LXQEN,Classes\CLSID\{B92E345D-F52D-41F3-B562-081BC772E3B9}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{B92E345D-F52D-41F3-B562-081BC772E3B9}\InProcServer32\,%SystemRoot%\system32\windowscodecs.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
12169,0001LXQEN,Classes\CLSID\{3050f6cd-98b5-11cf-bb82-00aa00bdce0b}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{3050f6cd-98b5-11cf-bb82-00aa00bdce0b}\InProcServer32\,C:\Windows\System32\iepeers.dll,REG_SZ,NT AUTHORITY\SYSTEM,
12170,0001LXQEN,Classes\CLSID\{b27b520e-46db-4720-b9c5-5f80acab23a4}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{b27b520e-46db-4720-b9c5-5f80acab23a4}\InProcServer32\,%SystemRoot%\system32\hgcpl.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,
12171,0001LXQEN,Classes\CLSID\{9456A480-E88B-43EA-9E73-0B2D9B71B1CA}\InProcServer32,HKEY_LOCAL_MACHINE\Software\Classes\CLSID\{9456A480-E88B-43EA-9E73-0B2D9B71B1CA}\InProcServer32\,%SystemRoot%\system32\windowscodecs.dll,REG_EXPAND_SZ,NT AUTHORITY\SYSTEM,


In [14]:
tasks[tasks['hostname'] == '0001LXQEN']

,accountlogontype,accountrunlevel,certificateissuer,certificatesubject,comment,creator,description,execarguments,execprogrampath,hostname,name,signatureexists,signatureverified,virtualpath
126,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_HIGHEST,Microsoft Windows Production PCA 2011,Microsoft Windows,,,The file is signed and the signature was verified.,-z,%windir%\system32\disksnapshot.exe,0001LXQEN,Diagnostics,true,true,\Microsoft\Windows\DiskFootprint\Diagnostics
127,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_LUA,Microsoft Windows Production PCA 2011,Microsoft Windows,$(@%SystemRoot%\system32\invagent.dll -702),$(@%SystemRoot%\system32\invagent.dll -701),The file is signed and the signature was verified.,-maintenance,%windir%\system32\compattelrunner.exe,0001LXQEN,ProgramDataUpdater,true,true,\Microsoft\Windows\Application Experience\ProgramDataUpdater
128,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_LUA,,,This task enrolls a certificate for Attestation Identity Key.,Microsoft Corporation,,,,0001LXQEN,AikCertEnrollTask,,,\Microsoft\Windows\CertificateServicesClient\AikCertEnrollTask
129,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_HIGHEST,,,Scans fault-tolerant volumes for fast crash recovery,Microsoft Corporation,,,,0001LXQEN,Data Integrity Scan for Crash Recovery,,,\Microsoft\Windows\Data Integrity Scan\Data Integrity Scan for Crash Recovery
130,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_HIGHEST,Microsoft Windows Production PCA 2011,Microsoft Windows,Optimizes the placement of data in storage tiers on all tiered storage spaces in the system.,Microsoft Corporation,The file is signed and the signature was verified.,-c -h -g -# -m 8 -i 13500,%windir%\system32\defrag.exe,0001LXQEN,Storage Tiers Optimization,true,true,\Microsoft\Windows\Storage Tiers Management\Storage Tiers Optimization
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,TASK_LOGON_GROUP,TASK_RUNLEVEL_LUA,,,This task applies color calibration settings.,Microsoft Corporation,,,,0001LXQEN,Calibration Loader,,,\Microsoft\Windows\WindowsColorSystem\Calibration Loader
249,TASK_LOGON_GROUP,TASK_RUNLEVEL_HIGHEST,,,The Windows Diagnostic Infrastructure Resolution host enables interactive resolutions for system problems detected by the Diagnostic Policy Service. It is triggered when necessary by the Diagnostic Policy Service in the appropriate user session. If the Diagnostic Policy Service is not running the task will not run,Microsoft Corporation,,,,0001LXQEN,ResolutionHost,,,\Microsoft\Windows\WDI\ResolutionHost
250,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_LUA,Microsoft Windows Production PCA 2011,Microsoft Windows,This task triggers a system reboot following update installation.,Microsoft Corporation,The file is signed and the signature was verified.,,%systemroot%\system32\MusNotification.exe,0001LXQEN,Reboot,true,true,\Microsoft\Windows\UpdateOrchestrator\Reboot
251,TASK_LOGON_SERVICE_ACCOUNT,TASK_RUNLEVEL_HIGHEST,Microsoft Windows Production PCA 2011,Microsoft Windows,Launch language cleanup tool,Microsoft Corporation,The file is signed and the signature was verified.,,%windir%\system32\lpremove.exe,0001LXQEN,LPRemove,true,true,\Microsoft\Windows\MUI\LPRemove


In [17]:
if 'hostname' in persistence_svc.columns and not persistence_svc.empty:
    print(persistence_svc[persistence_svc['hostname'] == '0001LXQEN'])
else:
    print("persistence_svc table is empty - no service-based persistence data available in this dataset.")

persistence_svc table is empty - no service-based persistence data available in this dataset.


In [18]:
print(persistence_reg[persistence_reg['hostname'] == '0001LXQEN'])

        hostname  \
6087   0001LXQEN   
6088   0001LXQEN   
6089   0001LXQEN   
6090   0001LXQEN   
6091   0001LXQEN   
...          ...   
12168  0001LXQEN   
12169  0001LXQEN   
12170  0001LXQEN   
12171  0001LXQEN   
12172  0001LXQEN   

                                                                               keypath  \
6087               Classes\CLSID\{4be5c3b3-53a2-4f2f-af9d-d26a5327eb92}\InProcServer32   
6088               Classes\CLSID\{ed9d80b9-d157-457b-9192-0e7280313bf0}\InProcServer32   
6089   Wow6432Node\Classes\CLSID\{3F037241-414E-11D1-A7CE-00A0C913F73C}\InProcServer32   
6090               Classes\CLSID\{46080CA7-7CB8-3A55-A72E-8E50ECA4D4FC}\InProcServer32   
6091               Classes\CLSID\{0C1A0EF4-B44F-4179-BDDD-31C27DE7A6D1}\InProcServer32   
...                                                                                ...   
12168              Classes\CLSID\{B92E345D-F52D-41F3-B562-081BC772E3B9}\InProcServer32   
12169              Classes\CLSID\{3050f

In [19]:
print(tasks[tasks['hostname'] == '0001LXQEN'])

               accountlogontype        accountrunlevel  \
126  TASK_LOGON_SERVICE_ACCOUNT  TASK_RUNLEVEL_HIGHEST   
127  TASK_LOGON_SERVICE_ACCOUNT      TASK_RUNLEVEL_LUA   
128  TASK_LOGON_SERVICE_ACCOUNT      TASK_RUNLEVEL_LUA   
129  TASK_LOGON_SERVICE_ACCOUNT  TASK_RUNLEVEL_HIGHEST   
130  TASK_LOGON_SERVICE_ACCOUNT  TASK_RUNLEVEL_HIGHEST   
..                          ...                    ...   
248            TASK_LOGON_GROUP      TASK_RUNLEVEL_LUA   
249            TASK_LOGON_GROUP  TASK_RUNLEVEL_HIGHEST   
250  TASK_LOGON_SERVICE_ACCOUNT      TASK_RUNLEVEL_LUA   
251  TASK_LOGON_SERVICE_ACCOUNT  TASK_RUNLEVEL_HIGHEST   
252            TASK_LOGON_GROUP      TASK_RUNLEVEL_LUA   

                         certificateissuer certificatesubject  \
126  Microsoft Windows Production PCA 2011  Microsoft Windows   
127  Microsoft Windows Production PCA 2011  Microsoft Windows   
128                                                             
129                                        